Evaluation and recommendations

In [66]:
#import library
import pandas as pd
import ast
import random
from collections import defaultdict

In [67]:
#load data that will be used here
train_baskets = pd.read_csv("../outputs/train_baskets.csv")
train_baskets["basket"] = train_baskets["basket"].apply(ast.literal_eval)

In [68]:
#load the rules
rules = pd.read_csv("../outputs/rules_filtered.csv")
rules["antecedents"] = rules["antecedents"].apply(ast.literal_eval)
rules["consequents"] = rules["consequents"].apply(ast.literal_eval)

print("train_baskets:", train_baskets.shape)
print("rules:", rules.shape)

train_baskets: (126408, 2)
rules: (179, 5)


In [69]:
# builing of the rule map, it works like antecedent -> [(consequent, confidence), ...]
# transform rule table in dictionnary
rule_map = defaultdict(list)
for _, r in rules.iterrows():
    ant = r["antecedents"][0]        # because 1->1 rules filter
    cons = r["consequents"][0]
    conf = float(r["confidence"])
    rule_map[ant].append((cons, conf))

# Sort by confidence with the best in first
for ant in rule_map:
    rule_map[ant].sort(key=lambda x: x[1], reverse=True)

In [70]:
OBS_RATIO = 0.7
antecedent_set = set(rule_map.keys())

def count_triggers(observed):
    return sum(1 for x in observed if x in antecedent_set)

rows = train_baskets.sample(n=min(2000, len(train_baskets)), random_state=42)

trigger_list = []
for _, row in rows.iterrows():
    basket = list(dict.fromkeys(row["basket"]))
    random.shuffle(basket)
    cut = max(1, int(len(basket) * OBS_RATIO))
    observed = set(basket[:cut])
    trigger_list.append(count_triggers(observed))

trigger_s = pd.Series(trigger_list)
print(trigger_s.describe())
print("pct observed with 0 triggers:", (trigger_s == 0).mean())
print("pct observed with >=1 trigger:", (trigger_s >= 1).mean())


count    2000.000000
mean        1.490500
std         1.867258
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max        12.000000
dtype: float64
pct observed with 0 triggers: 0.3935
pct observed with >=1 trigger: 0.6065


In [71]:
#fonction that make recom by taking observed product and tell the product recommanded, keep the best association
def recommend(observed, k=10):
    scores = defaultdict(float)

    # For each item you already have, look up rules item -> recommended_item
    for item in observed:
        for cons, conf in rule_map.get(item, []):
            # Don't recommend something already in observed
            if cons not in observed:
                # keep the best confidence score if multiple rules propose same cons
                scores[cons] = max(scores[cons], conf)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [pid for pid, _ in ranked[:k]]

In [72]:
#fonction tthat take a real basket and split in 2 part, keep = the one we know and hidden= the part we want to found
def split_basket(basket, observed_ratio=0.7, seed=None):
    if seed is not None:
        random.seed(seed)

    b = list(dict.fromkeys(basket))  # remove duplicates, keep order
    if len(b) < 2:
        return set(b), set()

    random.shuffle(b)
    cut = max(1, int(len(b) * observed_ratio))
    observed = set(b[:cut])
    hidden = set(b[cut:])
    return observed, hidden

In [73]:
# evaluation settings initialise
K = 10 #max 10 producsy recommand
OBS_RATIO = 0.7 # hidden 0.3 part of basket
N_SAMPLES = 2000 #test on 2000 baskets

# take random sample in the basket for evaluation
rows = train_baskets.sample(n=min(N_SAMPLES, len(train_baskets)), random_state=42)
metrics = []

for _, row in rows.iterrows():
    basket = row["basket"]

    # 1)split basket into observed and hidden
    observed, hidden = split_basket(basket, observed_ratio=OBS_RATIO)

    #if there is nothing to predict, skip
    if len(hidden) == 0:
        continue

    # 2)recommend top-K items using rules and observed items
    recs = recommend(observed, k=K)
    recs_set = set(recs)

    # 3)compare recommendations to hidden part, more hits better results
    hits = len(recs_set & hidden)

    # 4)compute metrics that will be use to evaluate the model
    hitrate = 1 if hits > 0 else 0 #
    precision = hits / K # verify if the recommendation is well done
    recall = hits / len(hidden) #verify if the hidden part is found by the prediction

    metrics.append({
        "order_id": row["order_id"],
        "basket_size": len(set(basket)),
        "observed_size": len(observed),
        "hidden_size": len(hidden),
        "nb_recommendations": len(recs),
        "hits": hits,
        "hitrate@K": hitrate,
        "precision@K": precision,
        "recall@K": recall
    })

In [74]:
#transform dictionary metrics into dataframes
metrics_df = pd.DataFrame(metrics)

#print exemple and average mean
display(metrics_df.head())

print("\nMean metrics:")
display(metrics_df[["hitrate@K", "precision@K", "recall@K"]].mean())

#how many recommendation are made, how many hits?
print("\nhow often we can recommend:")
display(metrics_df[["nb_recommendations", "hits"]].describe())


,order_id,basket_size,observed_size,hidden_size,nb_recommendations,hits,hitrate@K,precision@K,recall@K
0,1933506,5,3,2,1,0,0,0.0,0.0
1,2614445,7,4,3,1,0,0,0.0,0.0
2,25558,10,7,3,2,0,0,0.0,0.0
3,1793339,6,4,2,0,0,0,0.0,0.0
4,1826061,8,5,3,2,0,0,0.0,0.0



Mean metrics:


hitrate@K      0.103205
precision@K    0.010538
recall@K       0.030784
dtype: float64


how often we can recommend:


,nb_recommendations,hits
count,1841.000000,1841.000000
mean,1.173275,0.105378
std,1.199575,0.314121
min,0.000000,0.000000
25%,0.000000,0.000000
50%,1.000000,0.000000
75%,2.000000,0.000000
max,8.000000,2.000000


In [75]:
# save in outputs
metrics_df.to_csv("../outputs/metrics.csv", index=False)

In [76]:
display(metrics_df[["hitrate@K","precision@K","recall@K","nb_recommendations","hits"]].mean())
display(metrics_df["nb_recommendations"].value_counts().head(10))

hitrate@K             0.103205
precision@K           0.010538
recall@K              0.030784
nb_recommendations    1.173275
hits                  0.105378
dtype: float64

nb_recommendations
0    696
1    497
2    372
3    212
4     45
5     13
6      5
8      1
Name: count, dtype: int64

## Short summary and explanation

### Notebook goal
The goal is to evaluate our recommendation system based on **Apriori** on the **TRAIN** dataset.  
We keep part of the basket (**observed**) and hide the rest (**hidden**). Then we check whether the recommended products appear in the hidden part.

---

### Base settings I keep
I kept a simple configuration that runs well on my computer:
- **TOP_N_PRODUCTS = 3000**
- **MIN_SUPPORT = 0.002**
- Apriori with **low_memory=True** and **max_len=2**
- Simple **1 → 1** rules with:
  - **MIN_CONFIDENCE = 0.2**
  - **MIN_LIFT = 1.1**

---

### Average results with these rules
With these parameters, I obtain:
- **hitrate@10 ≈ 0.10**
- **precision@10 ≈ 0.011**
- **recall@10 ≈ 0.032**
- **nb_recommendations ≈ 1.18**

In short: the system works, but it often produces only **1 recommendation** (sometimes 0), so the scores remain low.

---

### Why do I have few recommendations?
I checked how many observed products actually trigger a rule (“triggers”):

- **Mean ≈ 1.50 triggers**: on average, only 1 to 2 observed items have an associated rule.
- **Median = 1**: half of the baskets have **at most 1 trigger**.
- **39.8% of baskets have 0 trigger**: in about 40% of cases, none of the observed items matches a rule antecedent, so we can't recommend anything.
- **60.2% of baskets have ≥ 1 trigger**: in about 60% of cases, at least one rule can be applied.

So, in many cases, there are **not enough rules** that apply to the basket, which explains the low number of recommendations.

---

### Tests I ran
I tried small changes:
- **confidence 0.2 → 0.18**: results did not improve (slightly worse)
- **lift 1.1 → 1.05**: almost no difference

Conclusion: these changes do not fix the main issue, because the real problem is the **rule coverage** (rules are triggered too rarely).

---

### Test with TOP_N_PRODUCTS = 5000
I tried 5000 products to get more rules, but Apriori became **too slow** (running for more than 15 minutes).  
So I went back to **3000**, which is more stable and faster.

---

### Final decision
I keep the base configuration (**3000 products**, support 0.002, max_len=2, confidence 0.2, lift 1.1), because:
- it runs correctly,
- it is easy to explain,
- and I can still build an application with these rules.

---

### Possible improvements if we want more recommendations
Later, we can improve without making it too complex:
- keep **1 → 1**, but keep more rules per product (e.g., Top 3 or Top 5)
- or use more complex rules (ex, **2 → 1**), but this can be heavier for the computer
